In [8]:
import requests
from bs4 import BeautifulSoup
from typing import Dict, Any, List
import time
from tqdm import tqdm 
import json

In [9]:
import RFA_utils

## with the help of RFA_utils we can use two main function
1. **extract_all_RFA_article_links: Extracts all article links from a given RFA webpage.**
2. scrape_rfa_article: Scrapes an article from the RFA website.


------------
-----------
------------
------------
------------

In [10]:
def loop_article_page(total_page, custom_url, key_code):
    """
    
    """
    return_file = {
        "Data": [],
        "message": "success",
        "response": 200
    }
    All_url_links = {}
    
    try:
        for i in tqdm(range(0, total_page)):
            final_url = custom_url + str(i*15) 
            found_url_links = RFA_utils.extract_all_RFA_article_links(final_url)
            key = key_code + str(i)
            All_url_links[key] = found_url_links
        return_file["Data"] = All_url_links
        return return_file
    
    except Exception as e:
        return_file["Data"] = All_url_links
        return_file["message"] = e
        return_file["response"] = 404
        return return_file

In [11]:
def check_error_in_links(All_url_link, page_code, print_each_error=False):
    """
    
    """

    error_counter = 0
    for page_id in range(1, len(All_url_link)):
        page_key = page_code + str(page_id)
        try:
            All_url_link.get(page_key)
            if  All_url_link.get(page_key)["Response"]!= 200:
                error_counter += 1
                if print_each_error:
                    print(page_key, All_url_link.get(page_key)["message"])
        except Exception as e:
            print(page_key, e)

    print(f"Total error in {page_code}: {error_counter}")

In [12]:
def save_json(path, file_name, data):
    """
    
    """
    with open(path+file_name, "w") as outfile:
        json.dump(data, outfile, indent=4)
        print(f"Successfully saved: {file_name}")

In [13]:
# Saving the final file
path = "./data/"

------------
------------
------------



# A. Extracting all Article links from ༸གོང་ས་མཆོག 
- Base url: https://www.rfa.org/tibetan/dalai-lama/story_archive?b_start:int=15
- Custom URL: https://www.rfa.org/tibetan/dalai-lama/story_archive?b_start:int= + str(i)
- Total page:116

In [14]:
total_page = 116 + 1
custom_url= "https://www.rfa.org/tibetan/dalai-lama/story_archive?b_start:int="
article_tag = "གོང་ས་མཆོག"
key_code = "Page " + article_tag + " "
print(f"Page code: {key_code}")

all_links = loop_article_page(total_page, custom_url, key_code)


Page code: Page གོང་ས་མཆོག 


100%|██████████| 117/117 [00:30<00:00,  3.82it/s]


In [15]:
print(f"Total page in {article_tag}: {len(all_links['Data'])}")

Total page in གོང་ས་མཆོག: 117


In [16]:
check_error_in_links(all_links['Data'], key_code, print_each_error=True)

Total error in Page གོང་ས་མཆོག : 0


In [24]:
# all_links

In [27]:
def compare_with_existing_data(new_data, existing_file_path, tag):
    """
    Compare newly extracted links with existing data to find new articles.
    
    Args:
        new_data (dict): Dictionary containing newly extracted article links
        existing_file_path (str): Path to the existing JSON file
        tag (str): Tag/category of the articles (e.g., "གོང་ས་མཆོག")
        
    Returns:
        dict: Dictionary containing statistics and new article links
    """
    comparison_result = {
        "tag": tag,
        "total_new_links": 0,
        "total_existing_links": 0,
        "new_links": [],
        "message": "Success",
        "response": 200
    }
    
    try:
        # Load existing data
        with open(existing_file_path, 'r', encoding='utf-8') as file:
            existing_data = json.load(file)
        
        # print(existing_data)
        
        # Extract all existing links into a set for faster lookup
        existing_links = set()
        for page_key in existing_data:
            # print(page_key)
            page_links = existing_data[page_key].get("Links", [])
            for link in page_links:
                existing_links.add(link)
        
        comparison_result["total_existing_links"] = len(existing_links)
        
        # Find new links
        new_links = []
        for page_key in new_data.get("Data", {}):
            page_links = new_data["Data"][page_key].get("Links", [])
            for link in page_links:
                if link not in existing_links:
                    new_links.append(link)
        
        comparison_result["total_new_links"] = len(new_links)
        comparison_result["new_links"] = new_links
        
        return comparison_result
    
    except Exception as e:
        comparison_result["message"] = f"Error comparing data: {str(e)}"
        comparison_result["response"] = 500
        return comparison_result

In [28]:
# Path to existing data file
existing_file_path = "./data/RFA_ALL_link_གོང་ས་མཆོག.json"

# Compare new data with existing data
comparison_result = compare_with_existing_data(all_links, existing_file_path, article_tag)

# Print comparison results
print(f"Existing links: {comparison_result['total_existing_links']}")
print(f"New links found: {comparison_result['total_new_links']}")

# If there are new links, you can save them or process them further
if comparison_result['total_new_links'] > 0:
    print("New articles found:")
    for i, link in enumerate(comparison_result['new_links'][:10]):  # Show first 10 new links
        print(f"{i+1}. {link}")
    
    if len(comparison_result['new_links']) > 10:
        print(f"... and {len(comparison_result['new_links']) - 10} more")
    
    # Option to save the new links to a separate file
    save_new_links = True  # Set to True if you want to save
    if save_new_links:
        new_links_file = f"./data/RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json"
        save_json("./data/", f"RFA_NEW_links_{article_tag}_{time.strftime('%Y%m%d')}.json", comparison_result)
else:
    print("No new articles found.")

Existing links: 1717
New links found: 140
New articles found:
1. https://www.rfa.org/tibetan/sargyur/hhdl-audience-430-devotees-rfatibetan-03192025061353.html
2. https://www.rfa.org/tibetan/sargyur/dalai-lama-gelong-vow-03182025065159.html
3. https://www.rfa.org/tibetan/sargyur/dalai-lama-audience-robert-thurmon-03172025055501.html
4. https://www.rfa.org/tibetan/sargyur/hhdl-congratulatory-message-cana-pm-mark-carney-03152025051629.html
5. https://www.rfa.org/tibetan/sargyur/hhdl-gelong-ordination-dharamsala-03152025041046.html
6. https://www.rfa.org/tibetan/sargyur/dalai-lama-teaching-jataka-dharamshala-03142025043815.html
7. https://www.rfa.org/tibetan/sargyur/hhdl-chotrul-duechen-2025-03142025015111.html
8. https://www.rfa.org/tibetan/sargyur/dalai-lama-gold-mercury-03132025151440.html
9. https://www.rfa.org/tibetan/sargyur/hhdl-gelong-ordination-2nd-day-51-devotees-03132025041517.html
10. https://www.rfa.org/tibetan/sargyur/hhdl-voice-for-the-voiceles-his-successor-born-free-world-